# 技能2 · Day 3 上机：人机协作治理 + 组织变革

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pandas** 分析人机协作审计日志，计算人工干预率/Agent自主完成率/人工修正率
2. 用 **matplotlib** 可视化任务完成时间分布，对比不同分工模式的效率
3. 用 **networkx** 构建组织协作网络，计算度中心性，发现桥接节点
4. 用 **McKinsey 7S 框架**评估组织AI就绪度，雷达图可视化薄弱维度
5. 用 **ADKAR 模型**诊断变革阻力，识别阻力最大的阶段
6. 用**天道推演**模拟组织变革阻力扩散路径，预判临界点

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：pandas（数据分析）+ matplotlib（可视化）+ networkx（网络分析）。
营销映射：营销团队导入AI Agent后的人机协作审计日志分析 + 组织变革评估。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> pandas / matplotlib / networkx 均为纯本地库，无需 API key，无需网络。

In [ ]:
# !pip install pandas matplotlib networkx -q

## 1. 场景背景与营销映射

**分析对象**：企业营销团队导入AI Agent后的人机协作审计日志。

**团队构成**：
- 营销策划师（人）-- 品牌战略、年度预算分配
- 文案生成Agent（AI）-- 小红书/朋友圈文案生成
- 合规审核员（人）-- 法规合规审核
- 投放优化Agent（AI）-- 广告出价、定向优化
- 数据分析师（人）-- 效果归因、用户洞察
- AI运营官（人，新角色）-- Agent监督、提示工程

**审计日志记录**：每次人机协作的完整过程，包括任务类型、执行者、是否有人工干预、干预类型、结果、耗时。

| TODO | 分析维度 | 工具 | 解决的问题 |
|------|---------|------|-----------|
| TODO1 | 审计日志统计 | pandas | 人工干预率/自主完成率/修正率 |
| TODO2 | 效率可视化 | matplotlib | 任务完成时间分布对比 |
| TODO3 | 组织网络分析 | networkx | 度中心性/桥接节点 |
| TODO4 | 组织就绪度 | McKinsey 7S | 7维度评分+雷达图 |
| TODO5 | 变革阻力诊断 | ADKAR | 5阶段评分+阻力识别 |
| TODO6 | 阻力推演 | 天道推演 | 阻力扩散路径+临界点 |

**数据说明**：审计日志基于真实文献参数（人工干预率15-30%，Stanford HAI/McKinsey报告）生成，非纯随机编造。

In [ ]:
import random
import json

# ============================================================
# 人机协作审计日志（基于真实文献参数生成）
# 参数来源：Stanford HAI AI Index / McKinsey AI状态报告
# 人工干预率15-30%为业界常见区间
# ============================================================

random.seed(42)  # 确保可复现

# 任务类型及其AI成熟度配置
# (task_type, ai_maturity, executor_dist, intervention_rate, duration_range)
TASK_CONFIG = {
    "文案生成": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.75, "Both": 0.20, "Human": 0.05},
        "intervention_rate": 0.18,  # AI高成熟度->低干预
        "duration_range": (30, 120),
    },
    "投放优化": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.80, "Both": 0.15, "Human": 0.05},
        "intervention_rate": 0.12,
        "duration_range": (10, 60),
    },
    "社媒运营": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.70, "Both": 0.20, "Human": 0.10},
        "intervention_rate": 0.15,
        "duration_range": (20, 90),
    },
    "用户分群": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.50, "Agent": 0.30, "Human": 0.20},
        "intervention_rate": 0.35,
        "duration_range": (120, 600),
    },
    "竞品分析": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.45, "Agent": 0.35, "Human": 0.20},
        "intervention_rate": 0.30,
        "duration_range": (300, 1200),
    },
    "效果归因": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.55, "Human": 0.30, "Agent": 0.15},
        "intervention_rate": 0.40,
        "duration_range": (600, 1800),
    },
    "营销策划": {
        "ai_maturity": "低",
        "executor_dist": {"Human": 0.75, "Both": 0.20, "Agent": 0.05},
        "intervention_rate": 0.85,
        "duration_range": (1800, 3600),
    },
    "合规审核": {
        "ai_maturity": "低",
        "executor_dist": {"Human": 0.90, "Both": 0.10, "Agent": 0.00},
        "intervention_rate": 0.95,
        "duration_range": (300, 900),
    },
}

INTERVENTION_TYPES = ["修正", "驳回", "指导", "无"]
OUTCOMES = ["success", "revised", "failed"]

def generate_audit_log(n=200):
    """生成人机协作审计日志"""
    logs = []
    task_types = list(TASK_CONFIG.keys())
    base_ts = 1717200000  # 2024-06-01 base timestamp

    for i in range(n):
        task_type = random.choice(task_types)
        cfg = TASK_CONFIG[task_type]

        # 选择执行者
        executor = random.choices(
            list(cfg["executor_dist"].keys()),
            weights=list(cfg["executor_dist"].values())
        )[0]

        # 是否有人工干预
        human_intervention = random.random() < cfg["intervention_rate"]

        # 干预类型
        if human_intervention:
            intervention_type = random.choices(
                ["修正", "驳回", "指导"],
                weights=[0.50, 0.20, 0.30]
            )[0]
        else:
            intervention_type = "无"

        # 结果
        if intervention_type == "驳回":
            outcome = random.choices(OUTCOMES, weights=[0.10, 0.60, 0.30])[0]
        elif intervention_type == "修正":
            outcome = random.choices(OUTCOMES, weights=[0.30, 0.65, 0.05])[0]
        else:
            outcome = random.choices(OUTCOMES, weights=[0.88, 0.10, 0.02])[0]

        # 耗时
        dur_min, dur_max = cfg["duration_range"]
        if executor == "Agent" and not human_intervention:
            duration = random.uniform(dur_min, dur_min + (dur_max - dur_min) * 0.3)
        elif executor == "Both":
            duration = random.uniform(dur_min, dur_max)
        else:
            duration = random.uniform(dur_min + (dur_max - dur_min) * 0.5, dur_max)

        timestamp = base_ts + i * 1800 + random.randint(0, 600)

        logs.append({
            "task_id": f"T{i+1:03d}",
            "task_type": task_type,
            "executor": executor,
            "agent_action": f"Agent执行{task_type}任务" if executor != "Human" else "无(Agent未参与)",
            "human_intervention": human_intervention,
            "intervention_type": intervention_type,
            "outcome": outcome,
            "duration_sec": round(duration, 1),
            "timestamp": timestamp,
        })

    return logs

# 生成审计日志
AUDIT_LOGS = generate_audit_log(200)

# 组织协作网络数据（基于McKinsey Agentic Organization模型）
# 节点 = 角色（人/Agent），边 = 协作关系
ORG_EDGES = [
    ("营销策划师", "文案生成Agent", 0.8),      # 策划师指导Agent
    ("文案生成Agent", "合规审核员", 0.7),        # Agent产出->人审核
    ("合规审核员", "营销策划师", 0.6),           # 审核反馈->策划
    ("营销策划师", "投放优化Agent", 0.5),       # 策划->投放指导
    ("投放优化Agent", "数据分析师", 0.7),       # 投放数据->分析
    ("数据分析师", "营销策划师", 0.8),           # 分析->策划反馈
    ("AI运营官", "文案生成Agent", 0.9),         # AI运营官监督Agent
    ("AI运营官", "投放优化Agent", 0.9),         # AI运营官监督Agent
    ("AI运营官", "营销策划师", 0.5),            # AI运营官<->策划师
    ("合规审核员", "AI运营官", 0.4),            # 审核<->AI运营
    ("数据分析师", "AI运营官", 0.5),            # 分析<->AI运营
    ("文案生成Agent", "投放优化Agent", 0.3),    # Agent间协作
    ("营销策划师", "数据分析师", 0.6),          # 策划<->分析
]

# McKinsey 7S 评估数据（1-5分，5分最佳）
SEVEN_S_SCORES = {
    "Strategy": 4,       # AI战略较清晰
    "Structure": 2,      # 组织结构未适配Agent（薄弱）
    "Systems": 3,        # 系统部分就绪
    "Shared Values": 3,  # 价值观转型中
    "Skills": 2,         # AI技能缺口大（薄弱）
    "Style": 3,          # 管理风格转型中
    "Staff": 4,          # 已招聘AI运营官
}

# ADKAR 变革阻力评估数据（1-5分，5分阻力最小/准备度最高）
ADKAR_SCORES = {
    "Awareness": 4,    # 员工已认知AI变革必要性
    "Desire": 2,       # 员工变革意愿低（阻力最大）
    "Knowledge": 3,    # 部分员工已接受AI培训
    "Ability": 2,      # 实操能力不足（阻力大）
    "Reinforcement": 3, # 巩固机制部分建立
}

# 组织成员变革阻力模型（用于天道推演）
# role: (初始阻力值0-1, 影响力0-1, 连接数)
ORG_MEMBERS = {
    "营销策划师": {"resistance": 0.3, "influence": 0.8, "connections": 3},
    "文案生成Agent": {"resistance": 0.0, "influence": 0.5, "connections": 3},
    "合规审核员": {"resistance": 0.6, "influence": 0.6, "connections": 3},
    "投放优化Agent": {"resistance": 0.0, "influence": 0.5, "connections": 3},
    "数据分析师": {"resistance": 0.4, "influence": 0.7, "connections": 4},
    "AI运营官": {"resistance": 0.1, "influence": 0.7, "connections": 4},
}

print("环境初始化完成")
print(f"审计日志: {len(AUDIT_LOGS)} 条记录")
print(f"组织网络: {len(ORG_EDGES)} 条协作关系")
print(f"7S评估: {len(SEVEN_S_SCORES)} 个维度")
print(f"ADKAR评估: {len(ADKAR_SCORES)} 个阶段")

## TODO 1：用 pandas 分析审计日志 -- 人工干预率/自主完成率/修正率

**pandas** 是 Python 数据分析事实标准。核心 API：
- `pd.DataFrame(data)`：从列表创建 DataFrame
- `df.groupby("col").agg(...)`：分组聚合
- `df["col"].value_counts()`：频率统计
- `df["col"].mean()`：均值

**关键指标定义**：
- 人工干预率 = 有干预的记录数 / 总记录数
- Agent自主完成率 = (executor=="Agent" 且 无干预) 的记录数 / 总记录数
- 人工修正率 = intervention_type=="修正" 的记录数 / 总记录数

**业界基准**（Stanford HAI / McKinsey）：人工干预率 15-30% 为业界常见区间。

In [ ]:
import pandas as pd

# 将审计日志转为 DataFrame
df = pd.DataFrame(AUDIT_LOGS)

# 计算核心指标
total = len(df)
intervention_rate = df["human_intervention"].mean()
autonomous_rate = ((df["executor"] == "Agent") & (~df["human_intervention"])).mean()
correction_rate = (df["intervention_type"] == "修正").mean()
rejection_rate = (df["intervention_type"] == "驳回").mean()
success_rate = (df["outcome"] == "success").mean()

# 按任务类型分组
by_task = df.groupby("task_type").agg(
    count=("task_id", "count"),
    intervention_rate=("human_intervention", "mean"),
    avg_duration=("duration_sec", "mean"),
    success_rate=("outcome", lambda x: (x == "success").mean()),
).round(3)

# 按执行者分组
by_executor = df.groupby("executor").agg(
    count=("task_id", "count"),
    intervention_rate=("human_intervention", "mean"),
    avg_duration=("duration_sec", "mean"),
).round(3)

audit_report = {
    "total_records": total,
    "intervention_rate": round(intervention_rate, 4),
    "agent_autonomous_rate": round(autonomous_rate, 4),
    "correction_rate": round(correction_rate, 4),
    "rejection_rate": round(rejection_rate, 4),
    "success_rate": round(success_rate, 4),
    "by_task_type": by_task.to_dict("index"),
    "by_executor": by_executor.to_dict("index"),
}

print("=== 人机协作审计报告 ===")
print(f"总记录数: {audit_report['total_records']}")
print(f"人工干预率: {audit_report['intervention_rate']:.1%} (业界基准: 15-30%)")
print(f"Agent自主完成率: {audit_report['agent_autonomous_rate']:.1%}")
print(f"人工修正率: {audit_report['correction_rate']:.1%}")
print(f"人工驳回率: {audit_report['rejection_rate']:.1%}")
print(f"任务成功率: {audit_report['success_rate']:.1%}")
print(f"\n--- 按任务类型 ---")
print(by_task.to_string())
print(f"\n--- 按执行者 ---")
print(by_executor.to_string())
print(f"\n审计报告: {audit_report}")

## TODO 2：用 matplotlib 可视化任务完成时间分布

**matplotlib** 是 Python 可视化基础库。核心 API：
- `plt.figure(figsize=(w,h))`：创建画布
- `plt.boxplot(data, labels=...)`：箱线图对比分布
- `plt.bar(x, y)`：柱状图
- `plt.title/plt.xlabel/plt.ylabel`：标签

**分析目标**：
1. 箱线图：对比 Agent / Human / Both 三种执行者的任务耗时分布
2. 柱状图：各任务类型的人工干预率对比

In [ ]:
import matplotlib
matplotlib.use("Agg")  # 无头环境兼容
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：箱线图 -- 三种执行者的任务耗时分布
executors = ["Agent", "Human", "Both"]
duration_data = [df[df["executor"] == e]["duration_sec"].values for e in executors]
bp = axes[0].boxplot(duration_data, labels=executors, patch_artist=True,
                     boxprops=dict(facecolor="lightblue"))
axes[0].set_title("Task Duration by Executor Type", fontsize=13)
axes[0].set_ylabel("Duration (seconds)")
axes[0].set_xlabel("Executor")

# 子图2：柱状图 -- 各任务类型的人工干预率
task_intervention = df.groupby("task_type")["human_intervention"].mean().sort_values(ascending=False)
colors = ["#e74c3c" if v > 0.5 else "#3498db" for v in task_intervention.values]
axes[1].barh(range(len(task_intervention)), task_intervention.values, color=colors)
axes[1].set_yticks(range(len(task_intervention)))
axes[1].set_yticklabels(task_intervention.index)
axes[1].set_xlabel("Human Intervention Rate")
axes[1].set_title("Intervention Rate by Task Type", fontsize=13)
axes[1].axvline(x=0.15, color="green", linestyle="--", label="15% baseline")
axes[1].axvline(x=0.30, color="orange", linestyle="--", label="30% baseline")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("task_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("图表已保存为 task_analysis.png")
print(f"\n耗时最长的执行者: {df.groupby('executor')['duration_sec'].mean().idxmax()}")
print(f"干预率最高的任务: {task_intervention.index[0]} ({task_intervention.values[0]:.1%})")

## 2. 组织协作网络分析

当 Agent 成为组织一等成员（Agentic Organization），组织结构从"树形"变为"网络"。networkx 可以量化分析这种网络结构。

**关键概念**：
- **度中心性（Degree Centrality）**：节点的连接数占比，值越高=协作枢纽
- **桥接中心性（Betweenness Centrality）**：节点处于最短路径上的频率，值越高=信息瓶颈
- **桥接节点**：连接不同子群的节点，移除后网络可能断裂

## TODO 3：用 networkx 构建组织协作网络

**networkx** 核心 API：
- `nx.Graph()`：创建无向图
- `G.add_node(name)` / `G.add_edge(a, b, weight=w)`：添加节点/边
- `nx.degree_centrality(G)`：度中心性
- `nx.betweenness_centrality(G)`：桥接中心性
- `nx.draw(G, with_labels=True)`：可视化网络

**分析目标**：识别组织中的协作枢纽和信息瓶颈（桥接节点）。

In [ ]:
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 构建组织协作网络
G = nx.Graph()
for a, b, w in ORG_EDGES:
    G.add_edge(a, b, weight=w)

# 计算度中心性和桥接中心性
deg_centrality = nx.degree_centrality(G)
betw_centrality = nx.betweenness_centrality(G, weight="weight")

# 识别桥接节点（betweenness最高的节点）
bridging_node = max(betw_centrality, key=betw_centrality.get)
# 识别枢纽节点（degree最高的节点）
hub_node = max(deg_centrality, key=deg_centrality.get)

# 网络基本属性
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
n_components = nx.number_connected_components(G)

# 可视化网络
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42, k=2)
# 节点颜色：Agent为绿色，人为蓝色
node_colors = ["#2ecc71" if "Agent" in n else "#3498db" for n in G.nodes()]
# 节点大小：按度中心性
node_sizes = [3000 * deg_centrality[n] + 500 for n in G.nodes()]
# 边宽度：按权重
edge_widths = [G[u][v]["weight"] * 3 for u, v in G.edges()]

nx.draw_networkx(G, pos, ax=ax, with_labels=True, node_color=node_colors,
                 node_size=node_sizes, font_size=9, font_weight="bold",
                 edge_color="gray", width=edge_widths, alpha=0.8)
ax.set_title("Organization Collaboration Network\n(Green=Agent, Blue=Human, Size=Degree Centrality)",
             fontsize=13)
plt.tight_layout()
plt.savefig("org_network.png", dpi=100, bbox_inches="tight")
plt.show()

network_report = {
    "n_nodes": n_nodes,
    "n_edges": n_edges,
    "density": round(density, 4),
    "n_components": n_components,
    "degree_centrality": {k: round(v, 4) for k, v in deg_centrality.items()},
    "betweenness_centrality": {k: round(v, 4) for k, v in betw_centrality.items()},
    "hub_node": hub_node,
    "bridging_node": bridging_node,
    "hub_degree": round(deg_centrality[hub_node], 4),
    "bridging_betweenness": round(betw_centrality[bridging_node], 4),
}

print("=== 组织协作网络分析 ===")
print(f"节点数: {network_report['n_nodes']} (角色)")
print(f"边数: {network_report['n_edges']} (协作关系)")
print(f"网络密度: {network_report['density']:.4f}")
print(f"连通分量: {network_report['n_components']}")
print(f"\n度中心性排序:")
for n, v in sorted(deg_centrality.items(), key=lambda x: -x[1]):
    print(f"  {n}: {v:.4f}")
print(f"\n桥接中心性排序:")
for n, v in sorted(betw_centrality.items(), key=lambda x: -x[1]):
    print(f"  {n}: {v:.4f}")
print(f"\n枢纽节点(度最高): {hub_node} (度={deg_centrality[hub_node]:.4f})")
print(f"桥接节点(中介最高): {bridging_node} (中介={betw_centrality[bridging_node]:.4f})")
print(f"\n网络报告: {network_report}")

## TODO 3 续：用 McKinsey 7S 框架评估组织AI就绪度

**McKinsey 7S 框架**：7个维度评估组织一致性（1-5分，5分最佳）。
- Strategy（战略）/ Structure（结构）/ Systems（系统）
- Shared Values（共同价值观）/ Skills（技能）/ Style（风格）/ Staff（人员）

**分析目标**：用雷达图可视化7S评分，识别薄弱维度（分数最低的维度是AI导入的瓶颈）。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# McKinsey 7S 雷达图
categories = list(SEVEN_S_SCORES.keys())
values = list(SEVEN_S_SCORES.values())
n = len(categories)

angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
values_closed = values + values[:1]
angles_closed = angles + angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles_closed, values_closed, "o-", linewidth=2, color="#2ecc71")
ax.fill(angles, values, alpha=0.25, color="#2ecc71")
ax.set_xticks(angles)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"], fontsize=9)
ax.set_title("McKinsey 7S Framework - AI Readiness Assessment", fontsize=14, pad=20)

# 标注每个维度的分数
for angle, value, cat in zip(angles, values, categories):
    ax.annotate(str(value), xy=(angle, value), fontsize=12, fontweight="bold",
                ha="center", va="bottom")

plt.tight_layout()
plt.savefig("seven_s_radar.png", dpi=100, bbox_inches="tight")
plt.show()

# 识别薄弱维度
sorted_scores = sorted(SEVEN_S_SCORES.items(), key=lambda x: x[1])
weakest = sorted_scores[:2]

seven_s_report = {
    "scores": SEVEN_S_SCORES,
    "average": round(np.mean(values), 2),
    "weakest_dimensions": [{"dimension": d, "score": s} for d, s in weakest],
    "strongest_dimension": sorted_scores[-1][0],
}

print("=== McKinsey 7S 组织就绪度评估 ===")
print(f"平均分: {seven_s_report['average']}/5.0")
print(f"\n各维度评分:")
for dim, score in sorted_scores:
    status = "薄弱" if score <= 2 else ("需改进" if score <= 3 else "良好")
    print(f"  {dim}: {score}/5 ({status})")
print(f"\n最薄弱维度: {weakest[0][0]}({weakest[0][1]}分), {weakest[1][0]}({weakest[1][1]}分)")
print(f"最强维度: {sorted_scores[-1][0]}({sorted_scores[-1][1]}分)")
print(f"\n7S评估报告: {seven_s_report}")

## TODO 4：用 ADKAR 模型诊断变革阻力

**ADKAR 模型**：5个阶段的变革准备度评估（1-5分，5分阻力最小）。
- Awareness（认知）/ Desire（意愿）/ Knowledge（知识）
- Ability（能力）/ Reinforcement（巩固）

**分析目标**：识别阻力最大的阶段（分数最低），设计针对性干预。

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ADKAR 柱状图
stages = list(ADKAR_SCORES.keys())
scores = list(ADKAR_SCORES.values())

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c" if s < 3 else ("#f39c12" if s == 3 else "#2ecc71") for s in scores]
bars = ax.bar(stages, scores, color=colors, edgecolor="black", linewidth=0.5)

# 在柱上标注分数
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.1,
            str(score), ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylim(0, 5.5)
ax.set_ylabel("Score (1=high resistance, 5=low resistance)", fontsize=11)
ax.set_title("ADKAR Change Management Assessment", fontsize=14)
ax.axhline(y=3, color="gray", linestyle="--", alpha=0.5, label="Threshold=3")
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("adkar_chart.png", dpi=100, bbox_inches="tight")
plt.show()

# 诊断阻力最大的阶段
weakest_stage = min(ADKAR_SCORES, key=ADKAR_SCORES.get)
weakest_score = ADKAR_SCORES[weakest_stage]

# 干预建议
INTERVENTIONS = {
    "Awareness": "全员AI变革沟通会，展示AI对业务的实际价值",
    "Desire": "设计AI增益激励方案，明确'AI增强而非替代'，提供转型培训",
    "Knowledge": "全员AI素养培训，选拔高潜力员工深度培训",
    "Ability": "设立AI实践社区(CoP)，提供实操指导和导师制",
    "Reinforcement": "建立AI使用KPI，定期复盘，分享成功案例",
}

adkar_report = {
    "scores": ADKAR_SCORES,
    "average": round(sum(scores) / len(scores), 2),
    "weakest_stage": weakest_stage,
    "weakest_score": weakest_score,
    "intervention": INTERVENTIONS[weakest_stage],
    "all_interventions": INTERVENTIONS,
}

print("=== ADKAR 变革阻力诊断 ===")
print(f"平均准备度: {adkar_report['average']}/5.0")
print(f"\n各阶段评分:")
for stage, score in sorted(ADKAR_SCORES.items(), key=lambda x: x[1]):
    status = "阻力大" if score <= 2 else ("需关注" if score == 3 else "准备充分")
    print(f"  {stage}: {score}/5 ({status})")
print(f"\n阻力最大阶段: {weakest_stage}({weakest_score}分)")
print(f"干预建议: {INTERVENTIONS[weakest_stage]}")
print(f"\nADKAR诊断报告: {adkar_report}")

## TODO 5：用天道推演模拟组织变革阻力扩散路径

**天道推演**：以天神视角俯视组织变革，模拟不同干预策略下阻力的演化路径。

**阻力扩散模型**：
- 每个成员有初始阻力值（0-1）和影响力（0-1）
- 阻力通过协作网络扩散：高阻力成员影响连接的低阻力成员
- 临界点：当平均阻力超过阈值（如0.5），变革可能失败

**推演步骤**：
1. 初始化各成员阻力值
2. 模拟N轮扩散（每轮：阻力通过边传播）
3. 记录每轮平均阻力，识别临界点
4. 对比"无干预"vs"干预AI运营官降阻力"两个场景

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 天道推演：模拟组织变革阻力扩散路径
N_EPOCHS = 12
DIFFUSION_RATE = 0.15  # 阻力扩散速率
DECAY_RATE = 0.92      # 自身阻力衰减率
THRESHOLD = 0.5        # 临界点阈值

def simulate_resistance(members_init, graph, n_epochs, intervention=None):
    """模拟阻力扩散"""
    members = {k: dict(v) for k, v in members_init.items()}
    history = []

    for epoch in range(n_epochs):
        # 应用干预（第3轮开始）
        if intervention and epoch == 3:
            for target, reduction in intervention.items():
                if target in members:
                    members[target]["resistance"] *= (1 - reduction)

        # 记录当前状态
        avg_resistance = np.mean([m["resistance"] for m in members.values()])
        max_resistance = max(m["resistance"] for m in members.values())
        history.append({
            "epoch": epoch,
            "avg_resistance": round(avg_resistance, 4),
            "max_resistance": round(max_resistance, 4),
            "members": {k: round(v["resistance"], 4) for k, v in members.items()},
        })

        # 阻力扩散
        new_resistances = {}
        for member in members:
            neighbors = list(graph.neighbors(member))
            if not neighbors:
                new_resistances[member] = members[member]["resistance"] * DECAY_RATE
                continue
            # 邻居阻力对当前成员的影响
            neighbor_influence = np.mean([
                members[n]["resistance"] * members[n]["influence"]
                for n in neighbors
            ])
            own = members[member]["resistance"] * DECAY_RATE
            diffused = neighbor_influence * DIFFUSION_RATE
            new_resistances[member] = min(1.0, own + diffused)

        for m in members:
            members[m]["resistance"] = new_resistances[m]

    return history

# 场景1：无干预
history_no_intervention = simulate_resistance(ORG_MEMBERS, G, N_EPOCHS)

# 场景2：干预（第3轮降低合规审核员和营销策划师的阻力）
intervention_plan = {"合规审核员": 0.5, "营销策划师": 0.3}
history_with_intervention = simulate_resistance(ORG_MEMBERS, G, N_EPOCHS, intervention_plan)

# 识别临界点
def find_critical_point(history, threshold):
    """找到平均阻力首次超过阈值的轮次"""
    for h in history:
        if h["avg_resistance"] > threshold:
            return h["epoch"]
    return None

cp_no = find_critical_point(history_no_intervention, THRESHOLD)
cp_with = find_critical_point(history_with_intervention, THRESHOLD)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：平均阻力演化
epochs = [h["epoch"] for h in history_no_intervention]
avg_no = [h["avg_resistance"] for h in history_no_intervention]
avg_with = [h["avg_resistance"] for h in history_with_intervention]

axes[0].plot(epochs, avg_no, "o-", color="#e74c3c", label="No Intervention", linewidth=2)
axes[0].plot(epochs, avg_with, "s-", color="#2ecc71", label="With Intervention", linewidth=2)
axes[0].axhline(y=THRESHOLD, color="gray", linestyle="--", alpha=0.5, label=f"Threshold={THRESHOLD}")
if cp_no is not None:
    axes[0].axvline(x=cp_no, color="#e74c3c", linestyle=":", alpha=0.3)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Average Resistance")
axes[0].set_title("Resistance Diffusion: Organization Level", fontsize=13)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 0.6)

# 子图2：各成员阻力演化（无干预场景）
for member in ORG_MEMBERS:
    member_history = [h["members"][member] for h in history_no_intervention]
    axes[1].plot(epochs, member_history, "o-", label=member, linewidth=1.5, markersize=4)
axes[1].axhline(y=THRESHOLD, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Resistance Level")
axes[1].set_title("Individual Resistance Evolution (No Intervention)", fontsize=13)
axes[1].legend(fontsize=8, loc="upper left")
axes[1].set_ylim(0, 0.8)

plt.tight_layout()
plt.savefig("resistance_simulation.png", dpi=100, bbox_inches="tight")
plt.show()

simulation_report = {
    "n_epochs": N_EPOCHS,
    "threshold": THRESHOLD,
    "no_intervention": {
        "final_avg_resistance": history_no_intervention[-1]["avg_resistance"],
        "critical_point": cp_no,
        "history": [(h["epoch"], h["avg_resistance"]) for h in history_no_intervention],
    },
    "with_intervention": {
        "final_avg_resistance": history_with_intervention[-1]["avg_resistance"],
        "critical_point": cp_with,
        "intervention_applied": intervention_plan,
        "history": [(h["epoch"], h["avg_resistance"]) for h in history_with_intervention],
    },
}

print("=== 天道推演：组织变革阻力扩散 ===")
print(f"模拟轮数: {N_EPOCHS}, 临界点阈值: {THRESHOLD}")
print(f"\n--- 场景1：无干预 ---")
print(f"最终平均阻力: {history_no_intervention[-1]['avg_resistance']:.4f}")
print(f"临界点(首次超阈值): {'第'+str(cp_no)+'轮' if cp_no else '未达到'}")
print(f"阻力演化: {[h['avg_resistance'] for h in history_no_intervention]}")

print(f"\n--- 场景2：干预(第3轮降低合规审核员和策划师阻力) ---")
print(f"干预方案: {intervention_plan}")
print(f"最终平均阻力: {history_with_intervention[-1]['avg_resistance']:.4f}")
print(f"临界点(首次超阈值): {'第'+str(cp_with)+'轮' if cp_with else '未达到(干预有效)'}")
print(f"阻力演化: {[h['avg_resistance'] for h in history_with_intervention]}")

print(f"\n--- 推演结论 ---")
if cp_no is not None and cp_with is None:
    print(f"干预有效：无干预场景在第{cp_no}轮突破临界点，干预后未突破")
elif cp_no is not None and cp_with is not None:
    print(f"干预延迟临界点：从第{cp_no}轮推迟到第{cp_with}轮")
else:
    print("两种场景均未突破临界点，组织变革可控")

print(f"\n天道推演报告: {simulation_report}")

## 3. 反思与前沿

### 反思问题
1. 审计日志显示哪类任务的人工干预率最高？根因是什么？（AI成熟度不足？任务复杂度过高？治理流程问题？）
2. 组织网络中的桥接节点是谁？如果这个人离职，网络会怎样？
3. McKinsey 7S 中哪个维度最薄弱？这对AI导入意味着什么？
4. ADKAR 中 Desire（意愿）分数最低，说明什么？如何提升？
5. 天道推演显示阻力扩散的临界点在第几轮？如何提前干预？

### 2026 前沿：Agentic Organization + Computer Use 审计
- **Agentic Organization**（McKinsey）：Agent成为组织一等成员，重塑工作定义/组织结构/治理体系
- **Computer Use / 计算机使用**：Agent直接操作GUI，审计日志需记录每步GUI操作（鼠标/键盘/截图）
- **天道推演×组织变革**：用多Agent仿真模拟组织成员群体动力学，预判阻力扩散路径
- **多Agent仿真**：将组织成员建模为Agent，模拟协作/冲突/博弈，预判组织变革结果

参考 [McKinsey Agentic Organization](https://www.mckinsey.com/capabilities/mckinsey-digital/our-insights/the-economic-potential-of-generative-ai) + [Stanford HAI AI Index](https://aiindex.stanford.edu/report/) + [Anthropic Computer Use](https://docs.anthropic.com/en/docs/build-with-claude/computer-use)。